In [1]:
import sys
import os

import numpy as np
import pandas as pd
from pathlib import Path
import platform
import psutil
import pprint

# ../bin/AnalysisVSPAERO.py をモジュールとしてインポート
sys.path.append(os.path.join('../../../')) # 親ディレクトリをモジュール探索パスに追加
from src.AnalysisVSPAERO import *

import openvsp as vsp

import os
print(f"Available CPUs: {os.cpu_count()}")


Available CPUs: 14


In [2]:

def get_system_info():
    info = {}
    info['OS'] = platform.system()
    info['OS Version'] = platform.version()
    info['Architecture'] = platform.machine()
    info['Processor'] = platform.processor()
    info['CPU Cores'] = psutil.cpu_count(logical=False)
    info['Logical CPU Cores'] = psutil.cpu_count(logical=True)
    info['Memory (Total)'] = f"{round(psutil.virtual_memory().total / (1024**3), 2)} GB"
    info['Memory (Available)'] = f"{round(psutil.virtual_memory().available / (1024**3), 2)} GB"
    
    return info

system_info = get_system_info()
pprint.pprint(system_info)

{'Architecture': 'AMD64',
 'CPU Cores': 12,
 'Logical CPU Cores': 14,
 'Memory (Available)': '5.17 GB',
 'Memory (Total)': '15.44 GB',
 'OS': 'Windows',
 'OS Version': '10.0.26200',
 'Processor': 'Intel64 Family 6 Model 170 Stepping 4, GenuineIntel'}


In [5]:

# Clear the current VSP model and read the new VSP file
vsp.ClearVSPModel()
vsp.Update()
vsp.ReadVSPFile(r"./SampleGlider.vsp3")
vsp.Update()

# Define the list of alpha angles, the Mach number and Reynolds number for the sweep
alpha = [0, 3, -3]
mach = [0.029]
reynolds = [4.3e5]

# Define the list of height
bref = 27
height = list((1-np.cos(np.linspace(0,1,12)*np.pi/2))*bref)[1:]

df = vsp_sweep_wig(
    vsp,
    alpha,
    mach,
    reynolds,
    height,
    analysis_method=0,
    trimmed=False,
    verbose=2,
    # ncpu=12,
    cmy_tolerance=1e-3,
    max_trim_evaluations=6,
    trim_step_deg=2.0,
    wake_num_iter=20,
)


VSPAEROComputeGeometry inputs

-> VSPAERO ground-effect sweep
 Mode          : polar
 OpenVSP       : OpenVSP 3.47.0
 VSP3          : c:/Users/mtkbirdman/OneDrive/Documents/OpenVSP/examples/notebooks/trimmed_ground_effect/./SampleGlider.vsp3
 Reference wing: WingGeom (PQFJJQUNWO)
 Geometry sets : loaded .vsp3 VSPAERO settings
   Thick [4]: ['FuselageGeom']
   Thin  [3]: ['WingGeom', 'HTailGeom', 'VTailGeom']
 VSPAERO refs  : Sref=18, bref=27, cref=0.625, Xcg=1.4, Ycg=0, Zcg=0, Symmetry=0, RefFlag=1
 Geometry ready : c:/Users/mtkbirdman/OneDrive/Documents/OpenVSP/examples/notebooks/trimmed_ground_effect/./SampleGlider.vspgeom (0.62 s)
 Cases         : 36
 VSPAERO run   : NCPU=4, WakeNumIter=20, FixedWakeFlag=False
 [  1/36] alpha=0 deg, Mach=0.029, Re=430000, ground=OFF, trimmed=False
   evaluation polar: alpha=0, Mach=0.029, Re=430000, ground=OFF, CGHeight=nan, elevator=None
 Alpha  Beta      CDi     CDiw      CDo    CDtot   CDwtot     CFix     CFiy     CFiz     CFox         CFoy     

In [6]:

df = make_CDo_correction(
  vsp, 
  df, 
  Weight=100, 
  xTr=(0.5, 0.7), 
  # CDpCL=0.0016, 
  # thickness=0.19, 
  # interference_factor=1.14, 
  altitude=0, 
  dT=0
)


In [7]:
df.to_csv('SampleGlider.GroundEffect.polar')
